# 04 — Đánh giá theo yêu cầu AI (S18)

Áp đúng quy tắc BR-10 của nền tảng: chuyển **đo tay** khi khoảng tin cậy rộng hơn 10 điểm hoặc chứa ngưỡng điều phối. Mục tiêu AIR-01…AIR-04 được đặt **trước khi đo**.

In [1]:
import sys, json
sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import soh_pipeline as S
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e4e3df'})
COL = {'B0005': '#2a78d6', 'B0006': '#eb6834', 'B0007': '#1baf7a', 'B0018': '#6250d6'}
m = json.load(open('../outputs/metrics.json', encoding='utf-8'))
pd.DataFrame([
  ['AIR-01', 'MAE ≤ 3,0 điểm', round(m['mae'], 2), m['pass']['AIR-01 MAE']],
  ['AIR-02', 'Độ phủ khoảng 90% ≥ 85%', round(m['coverage_90'], 3), m['pass']['AIR-02 coverage']],
  ['AIR-03', 'Độ chính xác quyết định tự động ≥ 97% (ngưỡng 70%)', round(m['auto_accuracy'], 3), m['pass']['AIR-03 auto accuracy']],
  ['AIR-04', 'Tỷ lệ chuyển đo tay ≤ 30%', round(m['manual_rate'], 3), m['pass']['AIR-04 manual rate']],
], columns=['Mã', 'Yêu cầu', 'Đo được', 'Đạt'])

,Mã,Yêu cầu,Đo được,Đạt
0,AIR-01,"MAE ≤ 3,0 điểm",2.530,True
1,AIR-02,Độ phủ khoảng 90% ≥ 85%,0.885,True
2,AIR-03,Độ chính xác quyết định tự động ≥ 97% (ngưỡng ...,0.871,False
3,AIR-04,Tỷ lệ chuyển đo tay ≤ 30%,0.946,False


In [2]:
pd.DataFrame({'Ngưỡng 70%': [m['manual_rate'], m['auto_accuracy'], m['accuracy_without_review']], 'Ngưỡng 80%': [m['sensitivity_threshold_80'][k] for k in ('manual_rate', 'auto_accuracy', 'accuracy_without_review')]}, index=['Tỷ lệ đo tay', 'Độ chính xác khi tự động', 'Độ chính xác nếu bỏ bước người duyệt']).round(3)

,Ngưỡng 70%,Ngưỡng 80%
Tỷ lệ đo tay,0.946,0.818
Độ chính xác khi tự động,0.871,1.000
Độ chính xác nếu bỏ bước người duyệt,0.786,0.962


**Kết luận.** Mô hình đạt yêu cầu độ chính xác (AIR-01, AIR-02) nhưng **chưa đạt yêu cầu vận hành** (AIR-03, AIR-04) trên dữ liệu demo: khoảng tin cậy còn rộng vì chỉ có 4 cell, và dữ liệu NASA dừng quanh 70% SoH — đúng vùng ngưỡng quyết định. Nếu bỏ bước người duyệt, độ chính xác phân loại ở ngưỡng 70% chỉ khoảng 79%. Quy tắc BR-10 làm đúng việc của nó: chặn quyết định tự động khi mô hình không chắc.

→ Quyết định DEC-11: ở MVP mô hình chạy **chế độ song song (shadow)**, mọi pin đo tay; chỉ bật quyết định tự động khi AIR-03 và AIR-04 đạt trên dữ liệu pack của VELIX.